# Processing Predicates

This notebook demonstrates how to load positive and negative predicates from pickle files, build a domain, and initialize a `LogicModel`.

## Strategy: Dense Core & Frequency-Based Probabilities

To handle a large number of predicates efficiently, we employ a "Dense Core" filtering strategy (keeping only the top 150 most frequent elements).

Furthermore, we demonstrate **Distributional Semantics** by using the **frequency** of assertions in the corpus to derive confidence scores. The hypothesis is: "The more often a fact is stated, the more confident we are in its truth."

We will:
1.  Count how many times each specific fact (e.g., `(Sanders, is_candidate, None)`) appears in the raw data.
2.  Normalize these counts against the most frequent fact for that subject.
3.  Use these normalized values as probabilities (0.0 to 1.0) in our LogicModel.

In [1]:
import pickle
import LogicModel as m
import numpy as np
import collections

## 1. Load and Filter Data

We prioritize the most frequent subjects (like "people", "movie") and explicitly ensure our key subject **"Sanders"** is included.

In [2]:
# Load ALL data
with open('Predicates/FILTERED-predicates.pickle', 'rb') as f:
    raw_pos = pickle.load(f)

with open('Predicates/FILTERED-negative_predicates.pickle', 'rb') as f:
    raw_neg = pickle.load(f)

# 1. Filter Pronouns
pronouns = {'he', 'she', 'it', 'him', 'her', 'they', 'them', 'we', 'us', 'you',
            'He', 'She', 'It', 'Him', 'Her', 'They', 'Them', 'We', 'Us', 'You',
            'this', 'This', 'that', 'That', 'these', 'These', 'those', 'Those', 
            'I', 'me', 'Me', 'my', 'My', 'myself', 'Myself'}

def filter_pronouns(pred_list):
    cleaned = []
    for s, p, o in pred_list:
        if s in pronouns: continue
        if o is not None and o in pronouns: continue
        cleaned.append((s, p, o))
    return cleaned

clean_pos = filter_pronouns(raw_pos)
clean_neg = filter_pronouns(raw_neg)

# 2. Identify Top Elements (The "Dense Core")
element_counts = collections.Counter()

for s, p, o in clean_pos + clean_neg:
    element_counts[s] += 1
    if o: element_counts[o] += 1

TOP_N = 150
top_elements = set([e for e, c in element_counts.most_common(TOP_N)])

# Explicitly ensure "Sanders" is in the domain
top_elements.add("Sanders")

print(f"Selected Top {TOP_N} frequent elements (plus 'Sanders').")

# 3. Final Filter (Keep duplicates for counting later!)
def keep_core(pred_list):
    final = []
    for s, p, o in pred_list:
        if s not in top_elements: continue
        if o is not None and o not in top_elements: continue
        final.append((s, p, o))
    return final

pos_predicates_raw = keep_core(clean_pos)
neg_predicates_raw = keep_core(clean_neg)

# Unique sets for building the model structure
# Custom sort key to handle None values in tuples
def safe_sort_key(t):
    return tuple((x if x is not None else "") for x in t)

pos_predicates_unique = sorted(list(set(pos_predicates_raw)), key=safe_sort_key)
neg_predicates_unique = sorted(list(set(neg_predicates_raw)), key=safe_sort_key)

print(f"Final Positive Predicates (Unique): {len(pos_predicates_unique)}")
print(f"Final Negative Predicates (Unique): {len(neg_predicates_unique)}")

Selected Top 150 frequent elements (plus 'Sanders').
Final Positive Predicates (Unique): 1950
Final Negative Predicates (Unique): 3593


### Check for Contradictions

We ensure that no predicate appears in both the positive and negative sets.

In [3]:
contradictions = set(pos_predicates_unique).intersection(set(neg_predicates_unique))
print(f"Number of Contradictions: {len(contradictions)}")

if contradictions:
    print("Sample Contradictions:", list(contradictions)[:5])

Number of Contradictions: 0


## 2. Process Data for LogicModel

We now organize these filtered predicates into the structures required by `LogicModel`.

In [4]:
domain_set = set()
unary_preds_dict = {}
binary_preds_dict = {}

def add_to_domain(elem):
    if elem is not None:
        domain_set.add(elem)

# Process Positive Predicates (True)
for subj, pred, obj in pos_predicates_unique:
    add_to_domain(subj)
    add_to_domain(obj)
    
    if obj is None:
        # Unary
        if pred not in unary_preds_dict:
            unary_preds_dict[pred] = []
        # Add as simple element (implies prob=1.0 by default, we will update later)
        unary_preds_dict[pred].append(subj)
    else:
        # Binary
        if pred not in binary_preds_dict:
            binary_preds_dict[pred] = []
        binary_preds_dict[pred].append((subj, obj))

# Process Negative Predicates (False)
for subj, pred, obj in neg_predicates_unique:
    add_to_domain(subj)
    add_to_domain(obj)
    
    if obj is None:
        # Unary
        if pred not in unary_preds_dict:
            unary_preds_dict[pred] = []
        # Add as tuple with prob=0.0
        unary_preds_dict[pred].append((subj, 0.0))
    else:
        # Binary
        if pred not in binary_preds_dict:
            binary_preds_dict[pred] = []
        # We do NOT add the pair to the list, so it defaults to False.

domain_list = sorted(list(domain_set))
print(f"Domain Size: {len(domain_list)}")
print(f"Number of Unary Predicates: {len(unary_preds_dict)}")
print(f"Number of Binary Predicates: {len(binary_preds_dict)}")

Domain Size: 150
Number of Unary Predicates: 2046
Number of Binary Predicates: 303


## 3. Build Logic Model

In [5]:
model = m.LogicModel(
    listOfElements=domain_list,
    dictionaryOfUnaryPredicates=unary_preds_dict,
    dictionaryOfBinaryPredicates=binary_preds_dict
)

model.buildAll()

## 4. Deriving Probabilities from Corpus Frequency

We now implement the "Distributional Semantics" idea: **Frequency = Confidence**.

We will look specifically at the subject **"Sanders"**.
1.  Count how many times each predicate appears for Sanders in the *raw* data.
2.  Find the maximum frequency.
3.  Calculate $P = \frac{\text{Count}}{\text{Max Count}}$.
4.  Update the model with these probabilities.

In [6]:
target = "Sanders"

# 1. Count Predicate Occurrences for Target
predicate_counts = collections.Counter()

print(f"--- Raw Facts about '{target}' ---")
for s, p, o in pos_predicates_raw:
    if s == target:
        # Create a unique key for the fact: (predicate, object)
        # If object is None, it's just (predicate,)
        key = (p, o)
        predicate_counts[key] += 1

# 2. Find Max Frequency
if predicate_counts:
    max_count = predicate_counts.most_common(1)[0][1]
    print(f"\nMax Count for any single fact: {max_count}\n")
    
    # 3. Derive Probabilities & Update Model
    for (pred, obj), count in predicate_counts.items():
        prob = count / max_count
        print(f"Fact: {pred}({target}, {obj}) | Count: {count} | Derived Prob: {prob:.2f}")
        
        if obj is None:
            # Unary Update
            model.updateUnaryPredicate(target, pred, prob)
        else:
            # Binary Update
            model.updateBinaryPredicate((target, obj), pred, prob)
else:
    print(f"No facts found for {target}")

--- Raw Facts about 'Sanders' ---

Max Count for any single fact: 2

Fact: calls(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: is_member(Sanders, None) | Count: 2 | Derived Prob: 1.00
Fact: is_candidate(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: want(Sanders, money) | Count: 1 | Derived Prob: 0.50
Fact: studied(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: went(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: led(Sanders, University) | Count: 1 | Derived Prob: 0.50
Fact: began(Sanders, career) | Count: 1 | Derived Prob: 0.50
Fact: lost(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: resigned(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: ran(Sanders, None) | Count: 2 | Derived Prob: 1.00
Fact: ran(Sanders, Smith) | Count: 1 | Derived Prob: 0.50
Fact: is_independent(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: became(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: voted(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: been(Sanders,

## 5. The Story of Sanders (Probabilistic)

Now we can tell a story about Sanders using these derived probabilities. Some facts are "certain" (relative to the corpus), while others are less certain.

We will explore the logic: **"Is Sanders a candidate AND a senator?"**

In [7]:
# Query 1: Is Sanders a candidate?
# This appeared relatively frequently in the data.
is_candidate = model.unaryOp("is_candidate", target)
print(f"1. Is '{target}' a 'candidate'?\n{is_candidate.flatten()}\n")

# Query 2: Is Sanders a senator?
# This appeared less frequently.
is_senator = model.unaryOp("is_senator", target)
print(f"2. Is '{target}' a 'senator'?\n{is_senator.flatten()}\n")

# Query 3: AND Operation
candidate_and_senator = model.andOp(is_candidate, is_senator)
print(f"3. [AND] Candidate AND Senator:\n{candidate_and_senator.flatten()}\n")

# Query 4: Binary Fact check
# Did Sanders deliver a speech? (Check binary probability if it exists)
delivered_speech = model.binaryOp("delivered", target, "speech")
print(f"4. Did '{target}' 'deliver' 'speech'?\n{delivered_speech.flatten()}\n")

print("Story Conclusion: The logic model now reflects the 'confidence' of the corpus.")

1. Is 'Sanders' a 'candidate'?
[0.5 0.5]

2. Is 'Sanders' a 'senator'?
[0.5 0.5]

3. [AND] Candidate AND Senator:
[0.25 0.75]

4. Did 'Sanders' 'deliver' 'speech'?
[0.5 0.5]

Story Conclusion: The logic model now reflects the 'confidence' of the corpus.
